# Lesson 23 - Gripper Challenges

A **challenge variant of Lesson 22**. There you learned how the robot sees an
object, drives to it, and picks it up. Here you complete a set of **gripper
tasks**, each a little harder than the last.

**You need a robot with the claw** to grab. On a claw-less robot the finding and
driving still run — the grab is skipped and the robot tells you. Each exercise
has starter code with `# TODO` lines for you to change. Get one working before
moving on.

## 1. Start The Robot

In [ ]:
from lesson_header import *
import time

show_v2_status()
myRobot = bot(base_speed=300)
myRobot.camera.center_all()

has_claw = myRobot.arm.available
print("Claw on this robot:", has_claw)

## Warm-up - Is the claw working?

Run the self-test and watch every step move before starting the challenges.

In [ ]:
myRobot.arm.test()

## Exercise 1 - Grab on command

**Goal:** place a light object in the open claw, then pick it up and put it back.
**Success check:** the claw closes on the object, lifts it, and sets it down
again without dropping it.

In [ ]:
# Get ready: open the claw at a middle height.
myRobot.arm.ready()
time.sleep(1)

# ---- Place an object in the open claw now ----

# TODO: pick it up, wait, then put it back down.
myRobot.arm.grab()
time.sleep(1)
myRobot.arm.place()

# TODO: if the claw closes before you are ready, slow it down:
#   myRobot.arm.grab(settle=1.0)

## Exercise 2 - Find it, drive to it, grab it

**Goal:** the robot finds a coloured object, drives up to it, and grabs it.
**TODO:** tune `GRAB_DISTANCE_CM` so the claw lands right on the object.
**Success check:** from ~40 cm away, the robot ends up holding the object.

In [ ]:
TARGET_COLOUR = "red"
GRAB_DISTANCE_CM = 12      # TODO: tune this (bigger = grabs from further away)
MAX_TRIES = 20

if has_claw:
    myRobot.arm.ready()

grabbed = False
for attempt in range(MAX_TRIES):
    decision = myRobot.vision.target_position(TARGET_COLOUR, deadzone=50, show=False)
    direction = decision["direction"]
    distance = myRobot.sonar.distance_cm()
    print(attempt, direction, distance)

    if direction == "lost":
        print("Lost the object - stopping.")
        break
    elif direction == "left":
        myRobot.move.left(seconds=0.15, speed=70)
    elif direction == "right":
        myRobot.move.right(seconds=0.15, speed=70)
    elif direction == "center":
        if distance is not None and distance <= GRAB_DISTANCE_CM:
            myRobot.stop()
            grabbed = myRobot.arm.grab() if has_claw else False
            print("Grabbed!" if grabbed else "Close enough (no claw / grab failed).")
            break
        else:
            myRobot.move.forward(seconds=0.3, speed=70)
    time.sleep(0.2)

myRobot.stop()
print("grabbed =", grabbed)

## Exercise 3 - Fetch and deliver

**Goal:** after grabbing (Exercise 2), carry the object to a "drop zone" and
place it there.
**TODO:** write the drive from the pick-up spot to the drop zone.
**Success check:** the object ends up in the drop zone, released by the claw.

In [ ]:
if grabbed:
    # TODO: drive to your drop zone. Change these moves to match your layout.
    myRobot.move.forward(seconds=1.0, speed=80)
    # myRobot.move.left(seconds=0.5, speed=80)   # e.g. turn toward the zone

    myRobot.arm.place()
    myRobot.stop()
    print("Delivered to the drop zone.")
else:
    print("Nothing grabbed yet - finish Exercise 2 first.")

## Exercise 4 - Pick only the target colour

**Goal:** with two or three different-coloured objects in view, grab **only** the
target colour and leave the others.
**TODO:** change `TARGET_COLOUR`, and make sure wrong colours are ignored.
**Success check:** the robot grabs the target colour and leaves the others alone.

In [ ]:
TARGET_COLOUR = "green"    # TODO: choose which colour to collect
GRAB_DISTANCE_CM = 12
MAX_TRIES = 20

# Because target_position only looks for TARGET_COLOUR, the other colours are
# ignored automatically. Reuse your Exercise 2 loop here and confirm it only
# reacts to TARGET_COLOUR.
if has_claw:
    myRobot.arm.ready()

for attempt in range(MAX_TRIES):
    decision = myRobot.vision.target_position(TARGET_COLOUR, deadzone=50, show=False)
    direction = decision["direction"]
    distance = myRobot.sonar.distance_cm()
    if direction == "lost":
        print("Target colour not in view.")
        break
    elif direction == "left":
        myRobot.move.left(seconds=0.15, speed=70)
    elif direction == "right":
        myRobot.move.right(seconds=0.15, speed=70)
    elif direction == "center":
        if distance is not None and distance <= GRAB_DISTANCE_CM:
            myRobot.stop()
            if has_claw:
                myRobot.arm.grab()
            print("Collected the", TARGET_COLOUR, "object.")
            break
        else:
            myRobot.move.forward(seconds=0.3, speed=70)
    time.sleep(0.2)

myRobot.stop()

## Stretch - Grab by image position (Hiwonder style)

Hiwonder's own grasping demo does **not** use the distance sensor. It watches how
far **down the camera image** the object is: `cy` is the object's vertical pixel
position, and it grows as the robot gets closer. When `cy` passes a set height
near the bottom of the frame, the object is close enough to grab.

**TODO:** pick a `PICK_Y` value and grab when `cy` passes it. Then compare: which
trigger - sonar distance or image position - copes better when the object is a
different size?

In [ ]:
TARGET_COLOUR = "red"
PICK_Y = 380               # TODO: tune (higher = lets the object get closer first)
MAX_TRIES = 20

if has_claw:
    myRobot.arm.ready()

for attempt in range(MAX_TRIES):
    decision = myRobot.vision.target_position(TARGET_COLOUR, deadzone=50, show=False)
    direction = decision["direction"]
    obj = decision["object"]                 # None when nothing is found
    cy = obj["cy"] if obj else None          # vertical pixel position of the object
    print(attempt, direction, "cy =", cy)

    if direction == "lost":
        print("Lost the object - stopping.")
        break
    elif direction == "left":
        myRobot.move.left(seconds=0.15, speed=70)
    elif direction == "right":
        myRobot.move.right(seconds=0.15, speed=70)
    elif direction == "center":
        if cy is not None and cy >= PICK_Y:
            myRobot.stop()
            if has_claw:
                myRobot.arm.grab()
            print("Grabbed using image position (cy =", cy, ")")
            break
        else:
            myRobot.move.forward(seconds=0.3, speed=70)
    time.sleep(0.2)

myRobot.stop()

## Done?

Which exercises worked first try? Which needed tuning? Write down the
`GRAB_DISTANCE_CM` and `PICK_Y` values that worked best for your robot and object
- they are different for every setup.

In [ ]:
myRobot.stop()